# 📗 RAPTOR: 원문을 묶고 요약 계층으로 검색하기

RAPTOR(Recursive Abstractive Processing for Tree-Organized Retrieval)는 비슷한 원문 청크를 묶어 GPT로 요약하고, 그 요약을 다시 묶어 요약한 트리를 검색에 씁니다. 작은 도서관 데모는 문장을 잎으로, PDF 따라하기는 긴 본문에서 직접 나눈 청크를 잎으로 사용합니다.

## 오늘의 목표

- [ ] 원문 청크를 임베딩하고 KMeans로 군집을 만들 수 있습니다.
- [ ] L1 요약문을 다시 임베딩·군집·요약해 L2를 만들 수 있습니다.
- [ ] 원문과 요약문을 함께 검색하고, 검색된 요약의 출처 청크와 PDF 페이지를 찾을 수 있습니다.

## ⏪ 지난 시간 복습

교안 01에서는 원문을 여러 방식으로 나누고, 작은 단위로 찾은 뒤 부모나 주변 문장을 돌려주었습니다. 돌려준 글은 모두 원문 조각이었습니다. 이번에는 **GPT가 쓴 요약문도 검색 후보**가 됩니다.

### 준비

새 커널에서도 시작할 수 있도록 교안 01의 경로·JSON 함수·모델 설정을 다시 실행합니다.

In [ ]:
# 파일·문서·표와 모델 연결에 필요한 도구부터 준비합니다.
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

#### 원문과 결과 파일의 경로

In [ ]:
# 실행할 노트북 폴더를 기준으로 입력 data와 생성 결과 output을 구분합니다.
import sys

material_dir = Path(".")
# 학생용·정답용 모두 실습자료 폴더의 util.py를 가져옵니다.
sys.path.insert(0, str(material_dir.resolve()))

data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)


def read_json(name):
    """data 폴더의 JSON 파일을 목록 또는 딕셔너리로 읽습니다."""
    return json.loads((data_dir / name).read_text(encoding="utf-8"))


def save_json(name, value):
    """처리 결과를 output 폴더에 한글을 유지해 저장합니다."""
    (output_dir / name).write_text(
        json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8"
    )

#### 모델 연결 설정

`.env`에 `OPENAI_API_KEY`를 넣습니다(`.env.example` 참고). 임베딩은 앞 단원과 같은 `text-embedding-3-large` 768차원, 요약·답변은 `gpt-5.6-luna`입니다(`OPENAI_CHAT_MODEL`로 바꿀 수 있습니다).

In [ ]:
# 현재 작업 폴더부터 상위로 .env를 찾아 키를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(".env에 OPENAI_API_KEY를 설정한 뒤 이 셀부터 다시 실행하세요.")

# 이 셀은 모델 설정만 준비합니다. GPT 요청은 뒤의 체인 invoke에서 발생합니다.
llm = ChatOpenAI(
    model=os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna"),
    use_responses_api=True,  # 요약·답변 요청에 OpenAI Responses API를 사용합니다.
)

# 문서와 질문을 같은 모델·차원으로 바꿔야 같은 인덱스에서 거리를 비교할 수 있습니다.
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large",
    dimensions=768,  # 문장·청크 하나를 768개 숫자로 표현합니다.
    check_embedding_ctx_length=False,  # 자동 길이 검사와 분할을 끄고 준비한 짧은 청크를 보냅니다.
)

print("모델 연결 설정 완료. 임베딩은 본문·질문을 벡터로 바꿀 때 요청합니다.")

#### 도서관 안내문(데모)과 인사 매뉴얼(따라하기) 읽기

도서관 데모는 제공된 짧은 문장을 사용합니다. PDF는 교안 01과 같은 긴 본문 네 편이며 `sentences` 목록이 없습니다. 따라하기에서는 **개념과 도입 본문 전체(PDF 9~20쪽)** 를 직접 청킹합니다. 요약 입력량을 제한하기 위해 연속 본문 한 편을 쓰며 질문에 맞는 문장만 골라 넣지 않습니다.

In [ ]:
# 원문 한 편이 기록 하나이며 text가 청킹할 본문입니다.
demo_records = read_json("demo_docs.json")
print("원문 수:", len(demo_records))
display(pd.DataFrame(demo_records)[["doc_id", "title"]])

In [ ]:
# 첫 원문을 읽고 문장과 조건이 어떻게 이어지는지 봅니다.
print(demo_records[0]["text"])

In [ ]:
# 연속된 PDF 페이지를 연결한 긴 본문을 읽습니다. 청킹은 뒤에서 직접 합니다.
practice_records = read_json("practice_docs.json")
print("원문 수:", len(practice_records))
display(pd.DataFrame(practice_records)[["doc_id", "title"]])

#### 원문을 LangChain Document로 바꾸기

`text`는 `page_content`에, 원문 ID·제목·위치는 `metadata`에 담습니다. 아직 자르지 않습니다.

In [ ]:
def make_documents(records):
    """원문 기록마다 본문과 출처(ID·제목·위치)를 담은 LangChain Document를 만듭니다."""
    return [
        Document(
            page_content=record["text"],
            # 분할한 뒤에도 source_id로 청킹 전 본문을 다시 찾습니다.
            metadata={
                "source_id": record["doc_id"],
                "title": record["title"],
                "url": record["url"],
                # 원문 전체 구간을 먼저 기록하고, 분할 뒤에는 각 청크의 구간으로 갱신합니다.
                "start_index": 0,
                "end_index": len(record["text"]),
            },
        )
        for record in records
    ]

#### 원문 위치와 PDF 페이지 지원 함수 가져오기

[util.py](util.py)의 두 함수는 청크의 끝 위치를 갱신하고, 해당 원문 구간과 겹치는 PDF 페이지를 붙입니다.

In [ ]:
# 시작 위치와 청크 길이로 끝 위치를 기록하는 지원 함수입니다.
from util import set_end_offsets

In [ ]:
# 원문 문자 구간을 PDF 페이지 범위와 연결하는 지원 함수입니다.
from util import add_pdf_pages

## 1. RAPTOR의 잎과 군집 만들기

도서관 데모는 문장 하나를 잎(L0)으로 사용합니다. 잎의 군집을 요약해 L1을 만들고, L1을 다시 묶어 요약해 L2를 만듭니다.

원논문은 UMAP으로 차원을 줄인 뒤 한 청크가 여러 군집에 속할 수 있는 확률 기반 군집을 쓰고, 군집 수도 데이터로 정합니다. 여기서는 앞 단원에서 배운 `KMeans`로 군집 수를 미리 정하고 한 원문 조각을 한 군집에만 넣습니다. [RAPTOR 논문](https://arxiv.org/abs/2401.18059)

<img style="background:#fff; max-width:100%; height:auto;" src="images/03_raptor.png" width="960" alt="잎(L0)을 군집·요약해 L1을 만들고, L1을 다시 군집·요약해 L2를 만드는 과정" />

| 필드 | 의미 |
|---|---|
| node_id | 노드 ID |
| level | 잎 0, 첫 요약 1, 그 위 요약 2 |
| child_ids | 바로 아래 입력 노드 |
| leaf_ids | 끝까지 내려가면 만나는 잎(출처) |

인사 매뉴얼 따라하기는 1절 → 2절 → 3절로 이어지므로 앞 따라하기를 마쳐야 다음 따라하기가 실행됩니다.

#### 문장 하나를 잎(L0) 하나로 만들기

In [ ]:
# 데이터에 미리 나눈 문장 하나를 잎(L0) 하나로 씁니다.
leaves = []

for record in demo_records:
    for sentence in record["sentences"]:
        # 문서가 바뀌어도 번호를 이어 붙여 전체 잎에서 ID가 겹치지 않게 합니다.
        node_id = f"leaf_{len(leaves)}"
        # 잎은 자식이 없고 출처(leaf_ids)는 자기 자신입니다. 제목은 요약 입력에서 문서를 구분합니다.
        leaves.append({
            "node_id": node_id,
            "level": 0,
            "text": sentence["text"],
            "child_ids": [],
            "leaf_ids": [node_id],
            "source": {"title": record["title"]},
        })

print("잎 수:", len(leaves))

`KMeans`는 유클리드 거리로 묶습니다. `normalize`로 벡터 길이(L2 노름)를 1로 맞춥니다. 정규화한 입력 벡터 사이에서는 유클리드 거리가 작을수록 코사인 유사도가 큽니다. 군집 번호는 `leaves`와 같은 순서로 나오므로 두 목록의 순서를 바꾸지 않습니다.

#### 같은 순서의 잎을 임베딩하기

In [ ]:
# 이 호출에서 잎 전체의 임베딩을 요청합니다. 군집 결과와 대응하도록 잎 순서를 유지합니다.
leaf_vectors = embedding_model.embed_documents(
    [node["text"] for node in leaves]
)

#### KMeans와 groupby로 잎을 3개 군집으로 묶기

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

# 벡터 길이를 1로 맞춰 크기 차이를 없앤 뒤 KMeans로 가까운 벡터를 묶습니다.
labels = KMeans(
    n_clusters=3,  # 묶을 군집의 개수입니다.
    random_state=0,  # 같은 입력에서 초기화에 쓰는 난수를 고정합니다.
    n_init=10,  # 초기화를 10번 시도해 군집 내 제곱거리 합이 가장 작은 결과를 선택합니다.
).fit_predict(normalize(leaf_vectors))

# labels의 순서는 입력 벡터와 같으므로 같은 순서의 노드에 군집 번호를 붙입니다.
node_table = pd.DataFrame({"cluster": labels, "node": leaves})
# 같은 군집 번호의 노드를 리스트로 모아 {군집 번호: 노드 리스트}를 만듭니다.
leaf_groups = node_table.groupby("cluster")["node"].agg(list).to_dict()

for group_id, members in leaf_groups.items():
    print("군집:", group_id, "/ 노드 수:", len(members))
    for node in members:
        print(node["node_id"], node["text"])

군집 번호는 이름표일 뿐 순서나 중요도가 아닙니다. 군집마다 원문을 읽고 어떤 주제가 함께 묶였는지 확인합니다.

#### 긴 PDF 본문을 직접 청킹하기

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 개념과 도입 본문 전체를 같은 입력으로 사용합니다. 다른 본문은 이 요약 실습에 넣지 않습니다.
practice_by_id = {record["doc_id"]: record for record in practice_records}
intro_records = [
    record for record in practice_records
    if record["doc_id"] == "hr_flexible_intro"
]
practice_documents = make_documents(intro_records)

# 교안 01에서 배운 Recursive로 문단·줄바꿈을 우선해 나눕니다.
practice_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,  # 기본 길이 기준은 문자 수이며, 청크는 최대 600자로 만듭니다.
    chunk_overlap=100,  # 경계의 문맥을 이어 주도록 인접 청크를 최대 100자 겹칩니다.
    strip_whitespace=False,  # 청크 앞뒤의 공백·줄바꿈을 유지합니다.
    add_start_index=True,  # 원문에서 청크의 시작 위치를 기록합니다.
)

# 분할로 달라진 끝 위치를 갱신한 뒤, 해당 문자 구간의 PDF 페이지 범위를 붙입니다.
practice_chunks = set_end_offsets(
    practice_splitter.split_documents(practice_documents)
)
practice_chunks = add_pdf_pages(practice_chunks, practice_by_id)

print("청킹 전 본문 수:", len(practice_documents), "/ 청킹 후 조각 수:", len(practice_chunks))
# 앞 세 청크에서 길이와 페이지 연결을 읽습니다. 전체 청크는 practice_chunks에 남아 있습니다.
for chunk in practice_chunks[:3]:
    print(chunk.metadata["pdf_start_page"], chunk.metadata["pdf_end_page"], len(chunk.page_content))
    print(chunk.page_content)

### 🖐️ 함께 따라하기: 직접 나눈 PDF 청크를 잎으로 만들고 군집화하기

`practice_chunks`로 데모와 같은 순서를 밟습니다. 잎은 문장으로 고정된 단위가 아니라, 이번에 검색·요약할 원문 청크입니다.

- **`practice_leaves`**: 데모 `leaves`와 같은 모양으로 청크마다 잎 하나(`leaf_0`부터)를 만듭니다. `text`는 `page_content`, `source`는 청크의 `metadata` 복사본입니다. 원문 ID·문자 위치·PDF 페이지 범위를 유지합니다.
- **`practice_leaf_vectors`**: 잎 text를 같은 순서로 임베딩합니다.
- **`practice_groups`**: 데모와 같은 설정(`normalize`, `n_clusters=3`, `random_state=0`, `n_init=10`)의 KMeans와 `groupby`로 {군집 번호: 잎 리스트}를 만듭니다.

**확인 기준**: `len(practice_leaves)`는 `len(practice_chunks)`와 같고, `practice_groups`의 키는 3개입니다. 군집 크기의 합은 `len(practice_leaves)`입니다. 군집마다 청크 2개를 읽어 주제가 모였는지 봅니다.

In [ ]:
# 1) practice_chunks의 각 청크로 잎을 만들고 metadata를 source에 복사하세요.
# 2) 잎 text를 임베딩해 practice_leaf_vectors에 담으세요.
# 3) KMeans와 groupby로 practice_groups를 만드세요.
# 여기에 코드를 작성하세요

### ✅ 바로 확인 퀴즈

**1. RAPTOR의 부모는 원문의 바로 인접한 부분끼리만 묶나요?**

<details><summary>정답 보기</summary><p>아닙니다. 임베딩으로 묶으므로 떨어진 문장이나 다른 문서도 같은 군집에 들어갈 수 있습니다.</p></details>

**2. KMeans 전에 `normalize`를 하는 이유는 무엇인가요?**

<details><summary>정답 보기</summary><p>길이의 영향을 줄이기 위해서입니다. 정규화한 입력 벡터 사이에서는 유클리드 거리가 작을수록 코사인 유사도가 큽니다.</p></details>

## 2. GPT 요약을 다시 요약해 두 계층 만들기

L1은 군집마다 원문 청크를 제목·ID와 함께 GPT에 넣어 요약합니다. L2는 **L1 요약문을 새로 임베딩**해 군집하고 그 요약문을 다시 요약합니다. 군집 수가 고정이라 요약 요청은 L1 3회와 L2 1회, 따라하기까지 8회입니다. 위층으로 갈수록 수치·예외가 빠지기 쉬우니 원문과 나란히 읽습니다.

#### 앞 단원의 Prompt·Model·Parser로 요약 체인 만들기

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 군집마다 context만 바꿔 쓰며, 서로 다른 원문의 조건을 섞지 않도록 요청합니다.
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "주어진 자료만 이용해 한국어로 3문장 이내로 요약하세요. "
               "서로 다른 항목의 조건을 섞지 말고 대상, 수치, 예외를 유지하세요. "
               "자료에 없는 사실을 보충하지 마세요."),
    ("human", "자료:\n{context}"),
])

# 다음 계층의 입력으로 사용할 수 있도록 응답을 요약 문자열로 받습니다.
summary_chain = summary_prompt | llm | StrOutputParser()

`summarize_groups`는 군집마다 GPT 요약 1회를 요청하고 출처 연결(`child_ids`·`leaf_ids`)을 붙입니다. L1·L2·따라하기에서 반복하므로 함수로 둡니다.

#### 군집별 요약과 출처 연결을 만드는 함수

In [ ]:
def summarize_groups(groups, level, summary_chain):
    """군집마다 GPT 요약을 한 번 요청해 요약 노드 리스트를 돌려줍니다."""
    summaries = []
    for group_id, members in sorted(groups.items()):
        sections = []
        for node in members:
            # 잎에는 원문 제목을 붙여 서로 다른 문서의 조건이 섞이지 않게 합니다. 요약 노드에는 제목이 없습니다.
            title = node["source"]["title"] if node["level"] == 0 else "하위 요약"
            sections.append(f"[{node['node_id']}] {title}\n{node['text']}")

        summary = summary_chain.invoke({"context": "\n\n".join(sections)})

        # 출처는 GPT에게 묻지 않고 입력 노드에서 모읍니다.
        # 이 실습에서는 노드가 한 군집에만 속해 leaf ID가 겹치지 않습니다. 청크 본문은 겹칠 수 있습니다.
        leaf_ids = [leaf_id for node in members for leaf_id in node["leaf_ids"]]

        # child_ids는 바로 아래 노드, leaf_ids는 계층을 내려갔을 때 도달하는 원문 잎입니다.
        summaries.append({
            "node_id": f"summary_{level}_{group_id}",
            "level": level,
            "text": summary,
            "child_ids": [node["node_id"] for node in members],
            "leaf_ids": leaf_ids,
        })

    return summaries

#### L0 군집을 요약해 L1 만들기

In [ ]:
# 잎의 군집마다 GPT를 한 번 호출합니다. child_ids는 그 군집의 입력 잎입니다.
level1 = summarize_groups(leaf_groups, level=1, summary_chain=summary_chain)

for node in level1:
    print(node["node_id"], "/ 자식:", node["child_ids"])
    print(node["text"])

#### L1 요약문을 새로 임베딩하고 한 군집으로 묶기

In [ ]:
# 두 번째 계층의 입력은 L1 요약문입니다. 잎의 벡터를 재사용하지 않습니다.
level1_vectors = embedding_model.embed_documents(
    [node["text"] for node in level1]
)

# L1이 3개뿐이라 군집 수를 1로 둡니다. 모두 한 군집이 되므로 이번에는 벡터가 결과를 바꾸지 않습니다.
# 벡터 길이를 1로 맞춰 크기 차이를 없앤 뒤 KMeans로 가까운 벡터를 묶습니다.
labels = KMeans(
    n_clusters=1,
    random_state=0,
    n_init=10,
).fit_predict(normalize(level1_vectors))

# labels의 순서는 입력 벡터와 같으므로 같은 순서의 노드에 군집 번호를 붙입니다.
node_table = pd.DataFrame({"cluster": labels, "node": level1})
# 같은 군집 번호의 노드를 리스트로 모아 {군집 번호: 노드 리스트}를 만듭니다.
level1_groups = node_table.groupby("cluster")["node"].agg(list).to_dict()

#### L1 군집을 요약해 L2를 만들고 잎도 보관하기

In [ ]:
# 모든 L1 요약을 합친 한 군집에서 최상위 요약 하나를 만듭니다.
level2 = summarize_groups(level1_groups, level=2, summary_chain=summary_chain)

# 요약을 추가해도 잎은 남겨 둡니다. 뒤에서 ID로 문장을 다시 조회합니다.
nodes = leaves + level1 + level2
nodes_by_id = {node["node_id"]: node for node in nodes}
save_json("library_raptor_nodes.json", nodes)  # 잎·L1·L2를 output 폴더에 저장해 열어 볼 수 있게 합니다.

print("계층별 노드 수:", len(leaves), len(level1), len(level2))
print(level2[0]["text"])

#### 대표 노드 세 개로 직접 입력과 출처 구분하기

In [ ]:
# 각 계층의 대표 하나씩을 봅니다. 세 행이 직접 연결된 하나의 경로라는 뜻은 아닙니다.
# L2의 직접 입력(child_ids)과 최종 출처(leaf_ids)가 어떻게 다른지 비교합니다.
display(
    pd.DataFrame([leaves[0], level1[0], level2[0]])[
        ["node_id", "level", "child_ids", "leaf_ids"]
    ]
)

L0는 자식이 없고 출처가 자기 자신입니다. L1의 두 목록은 모두 잎을 가리키지만, L2의 `child_ids`는 L1 요약문을, `leaf_ids`는 잎을 가리킵니다.

#### 한 요약의 입력과 출력 대조하기

In [ ]:
# child_ids로 실제 요약 입력을 찾아 주장·조건이 유지됐는지 읽습니다.
selected_summary = level1[0]  # 첫 요약을 관찰 대상으로 고른 것이며, 품질 순위는 아닙니다.

print("요약:", selected_summary["text"])
for child_id in selected_summary["child_ids"]:
    print("입력:", nodes_by_id[child_id]["text"])

요약의 주장 하나를 골라 어떤 입력 문장이 뒷받침하는지 찾으세요. 기간·횟수·조건이 빠졌거나 다른 규칙끼리 섞였다면 적어 둡니다. 프롬프트에 “수치, 예외를 유지하세요”라고 써도 보존 여부는 출력에서 확인해야 합니다.

### 🖐️ 함께 따라하기: 인사 매뉴얼 요약을 다시 요약하기

1절의 `practice_groups`에서 시작합니다.

- **`practice_level1`**: `practice_groups`를 `summarize_groups`로 요약합니다.
- **`practice_level1_vectors`**, **`practice_level1_groups`**: L1 text를 새로 임베딩해 군집 수 1로 묶습니다.
- **`practice_level2`**: 그 군집을 요약합니다.
- **`practice_nodes`**: `practice_leaves`·L1·L2를 이 순서로 합쳐 `hr_raptor_nodes.json`에 저장합니다.

**확인 기준**: `len(practice_nodes)`는 `len(practice_leaves) + 4`이고, `practice_level2[0]["child_ids"]`는 `['summary_1_0', 'summary_1_1', 'summary_1_2']`입니다.

In [ ]:
# 1) practice_groups를 요약해 practice_level1을 만드세요.
# 2) L1 text를 임베딩해 한 군집으로 묶고 요약해 practice_level2를 만드세요.
# 3) 잎·L1·L2를 합친 practice_nodes를 hr_raptor_nodes.json에 저장하세요.
# 여기에 코드를 작성하세요

### ✅ 바로 확인 퀴즈

**1. L2를 만들 때 임베딩할 글은 잎의 문장인가요?**

<details><summary>정답 보기</summary><p>아닙니다. L1 요약문입니다. 각 계층은 바로 아래 계층의 글을 새로 임베딩합니다.</p></details>

**2. 요약에 연결된 모든 leaf_ids를 정답 근거로 자동 인정해도 되나요?**

<details><summary>정답 보기</summary><p>안 됩니다. 연결은 출처 관계일 뿐입니다. 요약이 어떤 정보를 유지했는지는 내용으로 확인해야 합니다.</p></details>

## 3. 모든 계층을 검색하고 출처 원문으로 돌아가기

RAPTOR의 **collapsed tree 검색**은 잎과 모든 요약문을 한 인덱스에 넣고 질문과 가까운 노드를 고릅니다(위층부터 차례로 내려가며 찾지 않습니다). 어느 계층이 뽑힐지는 출력으로 확인합니다.

<img style="background:#fff; max-width:100%; height:auto;" src="images/04_raptor_search.png" width="960" alt="잎과 요약문을 한 인덱스에서 검색한 뒤, 검색된 요약의 leaf_ids로 잎을 찾아 답변 근거로 쓰는 흐름" />

RAPTOR는 검색된 원문·요약 노드를 답변 문맥으로 사용합니다. 이 실습에서는 조건 보존을 확인하려고 `leaf_ids`로 연결된 원문 잎을 찾아 답변에 사용합니다. [논문의 검색 방식](https://arxiv.org/html/2401.18059v1#S3) L2가 뽑히면 잎 전체가 딸려 오므로 전달 글자 수를 함께 봅니다.

#### 전체 계층을 검색할 Document 리스트로 바꾸기

In [ ]:
# 모든 계층을 같은 인덱스에서 검색하고 node_id로 원문 연결을 조회합니다.
collapsed_documents = [
    Document(
        page_content=node["text"],
        metadata={"node_id": node["node_id"], "level": node["level"]},
    )
    for node in nodes
]

#### 전체 계층을 Chroma에 저장하기

In [ ]:
from langchain_chroma import Chroma

# 문서 적재와 질문 검색에 같은 임베딩 설정을 사용합니다.
# add_documents가 넣을 문서 전체의 임베딩을 요청합니다. 돌려받는 ID 목록 대신 개수만 확인합니다.
raptor_store = Chroma(collection_name="day46_raptor_demo", embedding_function=embedding_model)
raptor_store.reset_collection()  # 다시 실행해도 같은 문서가 두 번 쌓이지 않도록 비웁니다.

added_ids = raptor_store.add_documents(collapsed_documents)
print("day46_raptor_demo 적재 수:", len(added_ids))

#### 구체적 질문과 넓은 질문의 결과 비교하기

In [ ]:
# 계층을 미리 선택하지 않습니다. 잎과 요약 전체에서 어떤 노드가 검색됐는지 확인합니다.
for question in [
    "그룹 학습실에 몇 분 늦으면 예약이 취소되나요?",
    "도서관 공간과 자료 이용 규칙 전반은?",
]:
    print("질문:", question)
    # k는 검색할 노드 수입니다. 잎·L1·L2에서 세 개씩 고르는 설정이 아닙니다.
    hits = raptor_store.similarity_search(question, k=3)

    for hit in hits:
        print("노드:", hit.metadata["node_id"], "/ 계층:", hit.metadata["level"])
        print(hit.page_content)

`trace_leaves`는 검색된 노드의 `leaf_ids`로 잎을 찾아 중복 없이 돌려줍니다(다시 검색하지 않고 ID로 조회만 합니다).

#### 검색 결과의 leaf_ids로 잎 조회하기

In [ ]:
def trace_leaves(hits, nodes_by_id):
    """검색된 노드들의 leaf_ids를 따라 원문 잎 노드를 중복 없이 돌려줍니다."""
    found = {}
    for hit in hits:
        node = nodes_by_id[hit.metadata["node_id"]]
        # L2도 L0를 바로 가리킵니다. 루트(L2)가 검색되면 잎 전체가 돌아옵니다.
        for leaf_id in node["leaf_ids"]:
            # 같은 ID만 한 번 남깁니다. 서로 다른 청크에 겹쳐 담긴 본문은 그대로 유지합니다.
            found[leaf_id] = nodes_by_id[leaf_id]

    return list(found.values())

#### 요약의 출처 원문 읽기

In [ ]:
# 검색된 노드가 요약이면 그 요약의 leaf_ids 전체가 출처로 따라옵니다.
question = "도서 대출 연장의 조건은 무엇인가요?"
# 두 노드를 찾더라도 요약에 연결된 출처 잎은 두 개보다 많을 수 있습니다.
raptor_hits = raptor_store.similarity_search(question, k=2)

for hit in raptor_hits:
    print("검색:", hit.metadata["node_id"], "/ 계층:", hit.metadata["level"])

evidence_leaves = trace_leaves(raptor_hits, nodes_by_id)

for node in evidence_leaves:
    print(node["node_id"], node["source"]["title"], node["text"])

print("출처 잎 수:", len(evidence_leaves), "/ 글자 수:",
      sum(len(node["text"]) for node in evidence_leaves))

#### 출처 원문으로 답하는 체인 만들기

In [ ]:
# context의 원문 앞에 붙인 [leaf_id]를 답변에서도 인용하도록 요청합니다.
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "제공된 원문 근거로만 한국어로 답하세요. 각 주장 뒤에 [leaf_id]를 붙이세요. "
               "근거에 답이 없으면 확인할 수 없다고 말하세요."),
    ("human", "질문: {question}\n\n원문 근거:\n{context}"),
])

# 답변 뒤에는 인용 ID의 존재뿐 아니라 원문이 주장을 뒷받침하는지 읽습니다.
answer_chain = answer_prompt | llm | StrOutputParser()

#### 출처 원문으로 GPT 답변 한 번 만들기

In [ ]:
# 답변에는 검색된 요약문 대신 연결된 잎 문장을 전달합니다.
context = "\n\n".join(
    f"[{node['node_id']}] {node['text']}"
    for node in evidence_leaves
)

# 프롬프트의 question·context 자리를 채우는 이 호출에서 GPT 답변을 요청합니다.
answer = answer_chain.invoke({"question": question, "context": context})
print(answer)

답변의 조건과 `[leaf_id]`를 위 문장과 대조하세요. 인용 ID가 붙어 있어도 그 문장이 주장을 뒷받침하는지는 따로 확인해야 합니다.

### 🖐️ 함께 따라하기: 인사 매뉴얼 요약 검색과 출처 확인

2절의 `practice_nodes`에서 시작합니다.

- **`practice_collapsed_documents`**, **`practice_raptor_store`**: 데모처럼 `practice_nodes`를 Document로 바꿔 컬렉션 `day46_raptor_practice`에 적재합니다.
- **`practice_raptor_hits`**: “유연근무 제도를 도입할 때 조사·승인·교육은 어떻게 준비하나요?”를 k=2로 검색합니다.
- **`practice_nodes_by_id`**, **`practice_evidence`**: {node_id: 노드} 딕셔너리와 `trace_leaves`로 출처 잎을 찾습니다.

**확인 기준**: 검색된 노드의 ID·계층과 `len(practice_evidence)`를 출력합니다. L2가 검색되면 `len(practice_evidence)`는 `len(practice_leaves)`와 같습니다. 요약이 검색되면 주장 하나를 출처 잎과 대조하고 조건 누락이 있으면 기록하세요. 없으면 없다고 적습니다. 잎만 검색되면 해당 잎의 조건과 PDF 페이지를 확인하세요.

In [ ]:
# 1) practice_nodes를 Document로 바꿔 practice_raptor_store에 적재하세요.
# 2) 질문을 k=2로 검색해 practice_raptor_hits에 담으세요.
# 3) trace_leaves로 practice_evidence를 찾고 검색 노드와 잎 수를 출력하세요.
# 여기에 코드를 작성하세요

### ✅ 바로 확인 퀴즈

**1. 검색 결과에 L2 요약이 들어오면 `trace_leaves`는 잎을 몇 개 돌려주나요?**

<details><summary>정답 보기</summary><p>전부입니다. L2의 leaf_ids는 모든 잎이라, 넓은 요약이 뽑힐수록 답변에 넣을 문장이 많아집니다.</p></details>

**2. 원문을 반환하는 Auto-merging과 RAPTOR의 요약은 무엇이 다른가요?**

<details><summary>정답 보기</summary><p>Auto-merging은 이미 있는 부모 원문을 반환합니다. RAPTOR는 모델이 새로 쓴 요약문을 인덱스에 추가합니다.</p></details>

## 이번 강의 정리

- 잎(L0)을 군집·요약해 L1을 만들고, L1 요약문을 다시 임베딩·군집·요약해 L2를 만들었습니다.
- 잎과 요약문을 한 인덱스에서 검색하고, `leaf_ids`로 찾은 출처 원문으로 요약과 답변의 조건을 확인했습니다.

## ⏭️ 다음 시간 예고

다음 시간에는 하이브리드 검색과 질의 변환을 다룹니다.